In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config


Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
# — Gold | Hype vs Reaction analysis table
from pyspark.sql import functions as F

# Load film master
df_master = spark.table(f"{SILVER_PATH}.film_master")

# — Build Hype Index
# Weighted combination: Trends peak (50%) + YouTube views (30%) + engagement (20%)
df_gold = (df_master
    .withColumn("hype_index",
        (F.col("hype_peak_score") * 0.50 +
         F.col("view_count_norm") * 0.30 +
         F.col("engagement_rate") * 0.20
        ).cast("double"))

    # — Hype Gap (core metric)
    .withColumn("hype_gap",
        (F.col("hype_index") - F.col("reaction_score")
        ).cast("double"))

    # — Verdict (calibrated to actual data range)
    .withColumn("verdict",
        F.when(F.col("hype_gap") >  5,   "Over-Hyped")
         .when(F.col("hype_gap") < -10,  "Underrated")
         .otherwise("Matched Expectations"))

    .select(
        "film",
        "release_date",
        "genre",
        "era",
        "runtime_minutes",
        "hype_peak_score",
        "view_count_norm",
        "engagement_rate",
        "hype_index",
        "imdb_rating",
        "reaction_score",
        "hype_gap",
        "verdict",
        "vote_count",
        "days_trailer_to_release"
    )
    .orderBy(F.col("hype_gap").desc())
)

print(f"✅ Gold table ready — {df_gold.count()} films")
print(f"\nVerdict breakdown:")
df_gold.groupBy("verdict").count().show()
display(df_gold)

✅ Gold table ready — 50 films

Verdict breakdown:
+--------------------+-----+
|             verdict|count|
+--------------------+-----+
|          Underrated|   38|
|Matched Expectations|   10|
|          Over-Hyped|    2|
+--------------------+-----+



film,release_date,genre,era,runtime_minutes,hype_peak_score,view_count_norm,engagement_rate,hype_index,imdb_rating,reaction_score,hype_gap,verdict,vote_count,days_trailer_to_release
Five Nights at Freddy's,2023-10-27,Horror,Short-Form Era (2015-2025),109,100,53.86899272987994,2.010464468399547,66.56279071264389,5.4,54.0,12.562790712643888,Over-Hyped,136255,122
Morbius,2022-04-01,Action,Short-Form Era (2015-2025),104,100,30.518112339983233,1.8375368671078327,59.52294107541653,5.1,51.0,8.522941075416533,Over-Hyped,170670,150
Inside Out 2,2024-06-14,Comedy,Short-Form Era (2015-2025),96,100,78.84788913943143,0.8650045607452432,73.82736765397847,7.5,75.0,-1.1726323460215298,Matched Expectations,246316,99
Joker,2019-10-04,Drama,Short-Form Era (2015-2025),122,100,100.0,1.8741069406040516,80.37482138812081,8.3,83.0,-2.6251786118791927,Matched Expectations,1680606,184
Black Panther,2018-02-16,Action,Short-Form Era (2015-2025),134,100,63.97590312774296,1.1173223442099147,69.41623540716488,7.3,73.0,-3.5837645928351236,Matched Expectations,904293,123
The Batman,2022-03-04,Action,Short-Form Era (2015-2025),176,100,76.27296763478036,2.210176220176816,73.32392553446948,7.8,78.0,-4.676074465530519,Matched Expectations,918937,139
It,2017-09-08,Horror,Short-Form Era (2015-2025),135,100,55.55894979876045,0.6348909355131579,66.79466312673077,7.3,73.0,-6.205336873269232,Matched Expectations,690781,43
Nope,2022-07-22,Horror,Short-Form Era (2015-2025),130,100,36.52637690010641,0.9144859176759702,61.14081025356712,6.8,68.0,-6.859189746432882,Matched Expectations,313333,159
The Flash,2023-06-16,Action,Short-Form Era (2015-2025),144,100,25.820836536524432,1.3357040029651297,58.01339176155035,6.6,65.99999999999999,-7.986608238449634,Matched Expectations,245890,124
Crazy Rich Asians,2018-08-15,Comedy,Short-Form Era (2015-2025),120,100,35.600217512102155,0.7510914389650722,60.83028354142366,6.9,69.0,-8.169716458576339,Matched Expectations,207773,114


In [0]:
# — Save to Gold Delta table
(df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD_PATH}.hype_gap_table")
)

print(f"✅ Saved to {GOLD_PATH}.hype_gap_table")
print(f"   Rows written: {df_gold.count()}")

display(spark.table(f"{GOLD_PATH}.hype_gap_table"))

✅ Saved to bootcamp_students.tiffani_ceresrain_gold.hype_gap_table
   Rows written: 50


film,release_date,genre,era,runtime_minutes,hype_peak_score,view_count_norm,engagement_rate,hype_index,imdb_rating,reaction_score,hype_gap,verdict,vote_count,days_trailer_to_release
Five Nights at Freddy's,2023-10-27,Horror,Short-Form Era (2015-2025),109,100,53.86899272987994,2.010464468399547,66.56279071264389,5.4,54.0,12.562790712643888,Over-Hyped,136255,122
Morbius,2022-04-01,Action,Short-Form Era (2015-2025),104,100,30.518112339983233,1.8375368671078327,59.52294107541653,5.1,51.0,8.522941075416533,Over-Hyped,170670,150
Inside Out 2,2024-06-14,Comedy,Short-Form Era (2015-2025),96,100,78.84788913943143,0.8650045607452432,73.82736765397847,7.5,75.0,-1.1726323460215298,Matched Expectations,246316,99
Joker,2019-10-04,Drama,Short-Form Era (2015-2025),122,100,100.0,1.8741069406040516,80.37482138812081,8.3,83.0,-2.6251786118791927,Matched Expectations,1680606,184
Black Panther,2018-02-16,Action,Short-Form Era (2015-2025),134,100,63.97590312774296,1.1173223442099147,69.41623540716488,7.3,73.0,-3.5837645928351236,Matched Expectations,904293,123
The Batman,2022-03-04,Action,Short-Form Era (2015-2025),176,100,76.27296763478036,2.210176220176816,73.32392553446948,7.8,78.0,-4.676074465530519,Matched Expectations,918937,139
It,2017-09-08,Horror,Short-Form Era (2015-2025),135,100,55.55894979876045,0.6348909355131579,66.79466312673077,7.3,73.0,-6.205336873269232,Matched Expectations,690781,43
Nope,2022-07-22,Horror,Short-Form Era (2015-2025),130,100,36.52637690010641,0.9144859176759702,61.14081025356712,6.8,68.0,-6.859189746432882,Matched Expectations,313333,159
The Flash,2023-06-16,Action,Short-Form Era (2015-2025),144,100,25.820836536524432,1.3357040029651297,58.01339176155035,6.6,65.99999999999999,-7.986608238449634,Matched Expectations,245890,124
Crazy Rich Asians,2018-08-15,Comedy,Short-Form Era (2015-2025),120,100,35.600217512102155,0.7510914389650722,60.83028354142366,6.9,69.0,-8.169716458576339,Matched Expectations,207773,114


In [0]:
spark.table(f"{GOLD_PATH}.hype_gap_table") \
    .groupBy("verdict", "era") \
    .count() \
    .orderBy("era", "verdict") \
    .show()

+--------------------+--------------+-----+
|             verdict|           era|count|
+--------------------+--------------+-----+
|          Underrated|Pre Short-Form|   12|
|Matched Expectations|Short-Form Era|    7|
|          Over-Hyped|Short-Form Era|    1|
|          Underrated|Short-Form Era|    8|
|Matched Expectations|    Transition|    3|
|          Underrated|    Transition|    7|
+--------------------+--------------+-----+



In [0]:
display(spark.table(f"{GOLD_PATH}.hype_gap_table"))

film,release_date,genre,era,runtime_minutes,hype_peak_score,view_count_norm,engagement_rate,hype_index,imdb_rating,reaction_score,hype_gap,verdict,vote_count,days_trailer_to_release
Morbius,2022-04-01,Action,Short-Form Era,104,100,30.51663903329016,1.8376418210514427,59.52252007419733,5.1,51.0,8.522520074197331,Over-Hyped,170670,150
Dune,2021-10-22,Sci-Fi,Short-Form Era,137,100,55.44752154452524,1.5686693832754857,66.94799034001267,6.2,62.0,4.947990340012666,Matched Expectations,192037,408
Joker,2019-10-04,Drama,Short-Form Era,122,100,100.0,1.8742187890655677,80.37484375781311,8.3,83.0,-2.6251562421868897,Matched Expectations,1680606,184
Black Panther,2018-02-16,Action,Transition,134,100,63.97229472587571,1.1173554584877077,69.41515950946027,7.3,73.0,-3.584840490539733,Matched Expectations,904293,123
The Batman,2022-03-04,Action,Short-Form Era,176,100,76.26764975358348,2.2103875091150837,73.32237242789806,7.8,78.0,-4.67762757210194,Matched Expectations,918937,139
It,2017-09-08,Horror,Transition,135,100,55.55659598102247,0.634902554558094,66.79395930521837,7.3,73.0,-6.206040694781635,Matched Expectations,690781,43
Nope,2022-07-22,Horror,Short-Form Era,130,100,36.525281644506464,0.9145113257927235,61.140486758510484,6.8,68.0,-6.859513241489516,Matched Expectations,313333,159
The Flash,2023-06-16,Action,Short-Form Era,144,100,25.819516673773222,1.3359078576330734,58.01303657365858,6.6,65.99999999999999,-7.986963426341404,Matched Expectations,245890,124
Crazy Rich Asians,2018-08-15,Comedy,Transition,120,100,35.59944465228087,0.7511125467940212,60.83005590504307,6.9,69.0,-8.169944094956932,Matched Expectations,207773,114
Indiana Jones and the Dial of Destiny,2023-06-30,Action,Short-Form Era,154,100,20.61137070931563,0.10465307796980045,56.20434182838865,6.5,65.0,-8.79565817161135,Matched Expectations,231222,84


In [0]:
spark.table(f"{GOLD_PATH}.hype_gap_table") \
    .groupBy("verdict", "era") \
    .count() \
    .orderBy("era", "verdict") \
    .show()

+--------------------+--------------------+-----+
|             verdict|                 era|count|
+--------------------+--------------------+-----+
|          Underrated|Pre Short-Form (2...|   19|
|Matched Expectations|Short-Form Era (2...|   10|
|          Over-Hyped|Short-Form Era (2...|    2|
|          Underrated|Short-Form Era (2...|   19|
+--------------------+--------------------+-----+



In [0]:
display(spark.table(f"{SILVER_PATH}.trends_combined"))

film,release_year,genre,era,pre_release_rating,post_release_rating,hype_drop,retention_pct
Batman Begins,2005,Action,Pre Short-Form (2005-2014),100,100,0.0,100.0
Brokeback Mountain,2005,Drama,Pre Short-Form (2005-2014),100,100,0.0,100.0
Casino Royale,2006,Action,Pre Short-Form (2005-2014),100,100,0.0,100.0
The Departed,2006,Drama,Pre Short-Form (2005-2014),100,100,0.0,100.0
Ratatouille,2007,Comedy,Pre Short-Form (2005-2014),100,100,0.0,100.0
No Country for Old Men,2007,Thriller,Pre Short-Form (2005-2014),100,100,0.0,100.0
There Will Be Blood,2007,Drama,Pre Short-Form (2005-2014),100,100,0.0,100.0
The Dark Knight,2008,Action,Pre Short-Form (2005-2014),100,100,0.0,100.0
Paranormal Activity,2009,Horror,Pre Short-Form (2005-2014),100,100,0.0,100.0
The Hangover,2009,Comedy,Pre Short-Form (2005-2014),100,100,0.0,100.0


In [0]:
display(spark.table(f"{SILVER_PATH}.trends_combined").select(
    "film",
    "release_year",
    "era",
    "genre",
    "pre_release_avg",
    "post_release_avg",
    "hype_drop",
    "retention_pct"
).orderBy("release_year"))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8825713052347213>, line 1
----> 1 display(spark.table(f"{SILVER_PATH}.trends_combined").select(
      2     "film",
      3     "release_year",
      4     "era",
      5     "genre",
      6     "pre_release_avg",
      7     "post_release_avg",
      8     "hype_drop",
      9     "retention_pct"
     10 ).orderBy("release_year"))

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:93, in Display.display_connect_table(self, df, **kwargs)
     88 except Exception as e:
     8